# Bronze layer: Equipment metadata Ingestion
**Purpose:** Load master equimpment data from CSV into Bronze delta table

**Input:** `/Volumes/dev/bronze/sample_data/equipment_metadata.csv`

**Output:** dev.bronze.equipment_metadata_raw

# Define Schema
Explicitly define the schema to ensure data types are correct

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

# Defind schema for equipment metadata
equipment_schema = StructType([
    StructField("equipment_id", StringType(), False),
    StructField("equipment_name", StringType(), False),
    StructField("equipment_type", StringType(), False),
    StructField("manufacturer", StringType(), True),
    StructField("model", StringType(), True),
    StructField("serial_number", StringType(), True),
    StructField("install_date", DateType(), True),
    StructField("factory_location", StringType(), True),
    StructField("production_line", StringType(), True),
    StructField("status", StringType(), True),
    StructField("criticality", StringType(), True),
    StructField("last_maintenance_date", DateType(), True)
])

print("Schema Defind")
print(f"Columns: {len(equipment_schema.fields)}")

# Read CSV files
Load the csv file with explicit schema

In [0]:
equipment_df = spark.read.csv("/Volumes/dev/bronze/sample_data/equipment_metadata.csv", header=True, schema=equipment_schema)

# Show basic stats
print(f"Records loaded: {equipment_df.count()}")
print("Schema")
equipment_df.printSchema()
print("Sample Data")
equipment_df.show(5, truncate=False)

# Add metadata columns
Add ingestion timestamp and source file tracking

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add metadata columns
equipment_enriched = equipment_df.withColumn("ingestion_timestamp",current_timestamp()) \
                     .withColumn("source_file",lit("equipment_metadata.csv")) \
                     .withColumn("data_source", lit("sample_data"))

print("Metadata columns added")
equipment_enriched.select("equipment_id", "source_file", "ingestion_timestamp").show(5)

# Write to bronze Delta Table
Save as managed Delta Table in Unity Catelog

In [0]:
# Configration
CATALOG = "dev"
SCHEMA = "bronze"
TABLE = "equipment_metadata_raw"
TABLE_FULL_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

print(f"Writing to: {TABLE_FULL_NAME}")

# Write to delta table
equipment_enriched.write \
    .format("Delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_FULL_NAME)

print(f"Data written to {TABLE_FULL_NAME}")

# Verify the Table 
Read back from Delta table and verify data

In [0]:
bronze_table = spark.read.table(TABLE_FULL_NAME)

#show stats
print(f"Table: {TABLE_FULL_NAME}")
print(f"Record Count: {bronze_table.count()}")
print("Sample Records")
bronze_table.show(5)

# Show table metadata
print("Table details")
spark.sql(f"DESCRIBE EXTENDED {TABLE_FULL_NAME}") \
    .filter("col_name IN ('Catalog','Table', 'Database', 'Location', 'Provider', 'Type')") \
    .show(truncate=False)

# Data Quality Check
verify data quality of loded data

In [0]:
# count by equipment type
print("Equipment by Type")
bronze_table.groupBy("equipment_type").count().orderBy("count", ascending=False).show(truncate=False)

# count by factory
print("Equipment by factory")
bronze_table.groupBy('factory_location').count().show()

# Count by Status
print("Equipment by status")
bronze_table.groupBy("status").count().show()

# Check the nulls in critical columns
print("Null check on equipment id")
null_count = bronze_table.filter(bronze_table.equipment_id.isNull()).count()
print(f"Null count: {null_count} (should be 0)")